# Testes do motor de dosimetria

Cobre o motor de dosimetria inteiro (`dosimetria/`): tipos de valor, as três fases e a dosimetria completa. Rode as células em ordem ("Run All"); qualquer `assert` que falhar interrompe a execução naquele ponto e mostra o traceback.

In [1]:
import sys
from pathlib import Path

for candidate in [Path.cwd(), *Path.cwd().parents]:
    if (candidate / "dosimetria").is_dir():
        sys.path.insert(0, str(candidate))
        break

from dosimetria import Fraction, Penalty, PenaltyRange

## Fraction

In [2]:
assert Fraction(1, 3).aplicar(360) == 120

# 1/3 de 365 = 121.66... -> desprezar a fração de dia (art. 11 do CP)
assert Fraction(1, 3).aplicar(365) == 121

assert Fraction(1, 2).aplicar(0) == 0

for numerador, denominador in [(1, 0), (1, -3)]:
    try:
        Fraction(numerador, denominador)
        raise AssertionError(f"deveria ter rejeitado denominador={denominador}")
    except ValueError:
        pass

try:
    Fraction(-1, 3)
    raise AssertionError("deveria ter rejeitado numerador negativo")
except ValueError:
    pass

assert str(Fraction(1, 6)) == "1/6"

print("Fraction: OK")

Fraction: OK


## Penalty

In [3]:
# convenção do projeto: 1 ano = 365 dias, 1 mês = 30 dias
assert Penalty.de_anos_meses_dias(anos=1).dias == 365
assert Penalty.de_anos_meses_dias(meses=1).dias == 30
assert Penalty.de_anos_meses_dias(anos=1, meses=2, dias=3).dias == 365 + 60 + 3

try:
    Penalty(-1)
    raise AssertionError("deveria ter rejeitado pena negativa")
except ValueError:
    pass

assert Penalty(360).mais(Fraction(1, 3)).dias == 480
assert Penalty(360).menos(Fraction(1, 6)).dias == 300

assert Penalty(100).mais_dias(50).dias == 150
assert Penalty(100).menos_dias(50).dias == 50

try:
    Penalty(10).menos_dias(20)
    raise AssertionError("deveria ter rejeitado pena negativa")
except ValueError:
    pass

assert Penalty.de_anos_meses_dias(anos=1, meses=4, dias=4).como_anos_meses_dias() == (1, 4, 4)

assert Penalty(100) < Penalty(200)
assert Penalty(200) > Penalty(100)
assert Penalty(100) <= Penalty(100)
assert Penalty(100) == Penalty(100)

assert str(Penalty.de_anos_meses_dias(anos=1, meses=4, dias=4)) == "1 ano, 4 meses, 4 dias"
assert str(Penalty(0)) == "0 dias"

# 1 ano = 365 dias, mas 12 meses = 360: os meses param em 11 e o excedente fica nos dias
assert Penalty(726).como_anos_meses_dias() == (1, 11, 31)
assert str(Penalty(726)) == "1 ano, 11 meses, 31 dias"
assert Penalty(364).como_anos_meses_dias() == (0, 11, 34)
assert Penalty(359).como_anos_meses_dias() == (0, 11, 29)
assert Penalty(360).como_anos_meses_dias() == (0, 11, 30)
assert Penalty(365).como_anos_meses_dias() == (1, 0, 0)
# a decomposição continua exata: volta para o mesmo número de dias
for total in range(0, 3 * 365):
    anos, meses, dias = Penalty(total).como_anos_meses_dias()
    assert meses <= 11
    assert Penalty.de_anos_meses_dias(anos, meses, dias).dias == total

print("Penalty: OK")

Penalty: OK


## PenaltyRange

In [4]:
faixa_furto_simples = PenaltyRange(
    minimo=Penalty.de_anos_meses_dias(anos=1),
    maximo=Penalty.de_anos_meses_dias(anos=4),
    origem="CP.art155",
)

assert faixa_furto_simples.contem(Penalty.de_anos_meses_dias(anos=2))
assert faixa_furto_simples.contem(faixa_furto_simples.minimo)
assert faixa_furto_simples.contem(faixa_furto_simples.maximo)
assert not faixa_furto_simples.contem(Penalty.de_anos_meses_dias(anos=5))

assert faixa_furto_simples.limitar(Penalty.de_anos_meses_dias(meses=6)) == faixa_furto_simples.minimo
assert faixa_furto_simples.limitar(Penalty.de_anos_meses_dias(anos=10)) == faixa_furto_simples.maximo

pena_dentro = Penalty.de_anos_meses_dias(anos=2)
assert faixa_furto_simples.limitar(pena_dentro) == pena_dentro

try:
    PenaltyRange(minimo=Penalty(100), maximo=Penalty(50), origem="teste")
    raise AssertionError("deveria ter rejeitado mínimo > máximo")
except ValueError:
    pass

print("PenaltyRange: OK")

PenaltyRange: OK


## Quantum (estratégias de incremento por circunstância)

In [5]:
from dosimetria import QuantumStrategy, IntervalFraction, MinimumFraction

faixa_furto = PenaltyRange(
    minimo=Penalty.de_anos_meses_dias(anos=1),
    maximo=Penalty.de_anos_meses_dias(anos=4),
    origem="CP.art155",
)

# intervalo = 3 anos = 1095 dias; 1/8 do intervalo
assert IntervalFraction().incremento_por_circunstancia(faixa_furto).dias == Fraction(1, 8).aplicar(1095)

# 1/6 do mínimo (365 dias)
assert MinimumFraction().incremento_por_circunstancia(faixa_furto).dias == Fraction(1, 6).aplicar(365)

assert isinstance(IntervalFraction(), QuantumStrategy)
assert "1/8" in IntervalFraction().nome

print("Quantum: OK")

Quantum: OK


## Fase 1 — pena-base (art. 59 do CP)

In [6]:
from dosimetria import JudicialCircumstance, Assessment, calcular_pena_base

TODAS_NEUTRAS = {c: Assessment.NEUTRA for c in JudicialCircumstance}


def com(desfavoraveis):
    valores = dict(TODAS_NEUTRAS)
    for circunstancia in desfavoraveis:
        valores[circunstancia] = Assessment.DESFAVORAVEL
    return valores


# nenhuma circunstância desfavorável -> pena-base no mínimo da faixa
resultado = calcular_pena_base(faixa_furto, TODAS_NEUTRAS, IntervalFraction())
assert resultado.pena_base == faixa_furto.minimo
assert resultado.passo.valor_antes == faixa_furto.minimo
assert resultado.passo.valor_depois == faixa_furto.minimo
assert resultado.passo.dispositivo == "CP.art155"

# uma circunstância desfavorável, estratégia 1/8 do intervalo
circunstancias_1 = com([JudicialCircumstance.CULPABILIDADE])
resultado_1 = calcular_pena_base(faixa_furto, circunstancias_1, IntervalFraction())
incremento_esperado = Fraction(1, 8).aplicar(1095)
assert resultado_1.pena_base.dias == faixa_furto.minimo.dias + incremento_esperado

# todas as 8 desfavoráveis nunca ultrapassa o máximo da faixa (mesmo com truncamento por fração)
todas_desfavoraveis = {c: Assessment.DESFAVORAVEL for c in JudicialCircumstance}
resultado_max = calcular_pena_base(faixa_furto, todas_desfavoraveis, IntervalFraction())
assert resultado_max.pena_base <= faixa_furto.maximo

# com um intervalo múltiplo de 8, 8 circunstâncias desfavoráveis batem exatamente no máximo
faixa_multipla_de_8 = PenaltyRange(minimo=Penalty(0), maximo=Penalty(800), origem="teste")
resultado_max_exato = calcular_pena_base(faixa_multipla_de_8, todas_desfavoraveis, IntervalFraction())
assert resultado_max_exato.pena_base == faixa_multipla_de_8.maximo

# circunstância faltando ou desconhecida é rejeitada
try:
    calcular_pena_base(faixa_furto, {JudicialCircumstance.CULPABILIDADE: Assessment.NEUTRA}, IntervalFraction())
    raise AssertionError("deveria exigir as 8 circunstâncias")
except ValueError:
    pass

print("Fase 1 (pena-base): OK")

Fase 1 (pena-base): OK


## Fase 2 — pena intermediária (agravantes e atenuantes, arts. 61 a 67)

In [7]:
from dosimetria import LegalCircumstance, CircumstanceDirection, calcular_pena_intermediaria

estrategia = IntervalFraction()
incremento = estrategia.incremento_por_circunstancia(faixa_furto).dias  # 136
pena_base_1_desfavoravel = resultado_1.pena_base  # 365 + 136 = 501


def agravante(codigo="reincidencia", preponderante=False):
    return LegalCircumstance(codigo, "CP.art61.I", CircumstanceDirection.AGRAVANTE, preponderante)


def atenuante(codigo="confissao_espontanea", preponderante=False):
    return LegalCircumstance(codigo, "CP.art65.III.d", CircumstanceDirection.ATENUANTE, preponderante)


# sem agravantes nem atenuantes -> pena intermediária = pena-base
sem_nada = calcular_pena_intermediaria(faixa_furto, pena_base_1_desfavoravel, [], estrategia)
assert sem_nada.pena_intermediaria == pena_base_1_desfavoravel

# uma agravante -> soma um incremento
so_agravante = calcular_pena_intermediaria(faixa_furto, pena_base_1_desfavoravel, [agravante()], estrategia)
assert so_agravante.pena_intermediaria.dias == pena_base_1_desfavoravel.dias + incremento

# uma atenuante que empurraria abaixo do mínimo -> Súmula 231, trava no mínimo
perto_do_minimo = calcular_pena_intermediaria(faixa_furto, faixa_furto.minimo, [atenuante()], estrategia)
assert perto_do_minimo.pena_intermediaria == faixa_furto.minimo

# agravante preponderante + atenuante não preponderante -> só a agravante conta (art. 67)
concurso_agravante_prepondera = calcular_pena_intermediaria(
    faixa_furto,
    pena_base_1_desfavoravel,
    [agravante(preponderante=True), atenuante(preponderante=False)],
    estrategia,
)
assert concurso_agravante_prepondera.pena_intermediaria.dias == pena_base_1_desfavoravel.dias + incremento
assert "preponderam as agravantes" in concurso_agravante_prepondera.passo.motivo

# atenuante preponderante + agravante não preponderante -> só a atenuante conta (art. 67)
concurso_atenuante_prepondera = calcular_pena_intermediaria(
    faixa_furto,
    pena_base_1_desfavoravel,
    [agravante(preponderante=False), atenuante(preponderante=True)],
    estrategia,
)
assert concurso_atenuante_prepondera.pena_intermediaria.dias == pena_base_1_desfavoravel.dias - incremento

# nenhuma das duas é preponderante -> compensação líquida (uma cancela a outra)
compensacao = calcular_pena_intermediaria(
    faixa_furto, pena_base_1_desfavoravel, [agravante(), atenuante()], estrategia
)
assert compensacao.pena_intermediaria == pena_base_1_desfavoravel

print("Fase 2 (pena intermediária): OK")

Fase 2 (pena intermediária): OK


## Fase 3 — pena definitiva (causas de aumento e diminuição, art. 68)

In [8]:
from dosimetria import (
    ModifyingCause,
    Composition,
    CauseDirection,
    CauseOrigin,
    calcular_pena_definitiva,
)

AUMENTO, DIMINUICAO = CauseDirection.AUMENTO, CauseDirection.DIMINUICAO
GERAL, ESPECIAL = CauseOrigin.PARTE_GERAL, CauseOrigin.PARTE_ESPECIAL

repouso_noturno = ModifyingCause("repouso_noturno", "CP.art155.§1", AUMENTO, ESPECIAL, Fraction(1, 3))


def tentativa(escolhida=None, justificativa=None):
    return ModifyingCause(
        "tentativa", "CP.art14.parágrafo_único", DIMINUICAO, GERAL,
        fracao_min=Fraction(1, 3), fracao_max=Fraction(2, 3),
        fracao_escolhida=escolhida, justificativa=justificativa,
    )


# sem causas -> pena definitiva = pena intermediária, com um passo explicando
sem_causas = calcular_pena_definitiva(Penalty(365), [], Composition.CASCATA)
assert sem_causas.aplicando_todas.pena_definitiva == Penalty(365)
assert len(sem_causas.aplicando_todas.passos) == 1
assert sem_causas.limitada_art68 is None

# furto noturno: 365 + 1/3 = 486,67 -> 486 (art. 11)
noturno = calcular_pena_definitiva(Penalty(365), [repouso_noturno], Composition.CASCATA)
assert noturno.aplicando_todas.pena_definitiva == Penalty(486)
assert noturno.aplicando_todas.passos[0].dispositivo == "CP.art155.§1"

# 3ª fase pode ultrapassar o máximo da faixa...
acima = calcular_pena_definitiva(faixa_furto.maximo, [repouso_noturno], Composition.CASCATA)
assert acima.aplicando_todas.pena_definitiva == Penalty(1946)
assert not faixa_furto.contem(acima.aplicando_todas.pena_definitiva)

# ...e ficar abaixo do mínimo (tentativa, fração mínima por padrão: 365 - 1/3 = 243,33 -> 243)
abaixo = calcular_pena_definitiva(faixa_furto.minimo, [tentativa()], Composition.CASCATA)
assert abaixo.aplicando_todas.pena_definitiva == Penalty(243)
assert not faixa_furto.contem(abaixo.aplicando_todas.pena_definitiva)

# fração acima da mínima sem justificativa é rejeitada; fora do intervalo legal também
for escolhida, justificativa in [(Fraction(2, 3), None), (Fraction(2, 3), "   "), (Fraction(3, 4), "qualquer")]:
    try:
        tentativa(escolhida, justificativa)
        raise AssertionError(f"deveria ter rejeitado {escolhida} com justificativa={justificativa!r}")
    except ValueError:
        pass

# com justificativa, a fração escolhida vale e a justificativa vai para o relatório
justificada = calcular_pena_definitiva(
    Penalty(365), [tentativa(Fraction(2, 3), "iter criminis mal iniciado")], Composition.CASCATA
)
assert justificada.aplicando_todas.pena_definitiva == Penalty(121)  # 365/3 = 121,67
assert "iter criminis mal iniciado" in justificada.aplicando_todas.passos[0].motivo

# cascata x sobre a pena intermediária: +1/3 e -1/3 sobre 360
causas_mistas = [repouso_noturno, tentativa()]
assert calcular_pena_definitiva(Penalty(360), causas_mistas, Composition.CASCATA).aplicando_todas.pena_definitiva == Penalty(320)
assert calcular_pena_definitiva(Penalty(360), causas_mistas, Composition.SOBRE_PENA_INTERMEDIARIA).aplicando_todas.pena_definitiva == Penalty(360)

# arredonda uma vez só, no fim da fase: 365 * 7/6 * 7/6 = 496,8 -> 496
# (truncando a cada passo daria 425 -> 495); por isso a ordem das causas não importa
concurso_formal = ModifyingCause("concurso_formal", "CP.art70", AUMENTO, GERAL, Fraction(1, 6), Fraction(1, 2))
continuidade = ModifyingCause("crime_continuado", "CP.art71", AUMENTO, GERAL, Fraction(1, 6), Fraction(2, 3))
duas = calcular_pena_definitiva(Penalty(365), [concurso_formal, continuidade], Composition.CASCATA)
invertidas = calcular_pena_definitiva(Penalty(365), [continuidade, concurso_formal], Composition.CASCATA)
assert duas.aplicando_todas.pena_definitiva == Penalty(496)
assert invertidas.aplicando_todas.pena_definitiva == Penalty(496)
assert duas.limitada_art68 is None  # causas da Parte Geral não entram no art. 68, parágrafo único

# os passos encadeiam: o "depois" de um é o "antes" do seguinte
passos = duas.aplicando_todas.passos
assert passos[0].valor_antes == Penalty(365)
assert passos[-1].valor_depois == duas.aplicando_todas.pena_definitiva
assert all(a.valor_depois == b.valor_antes for a, b in zip(passos, passos[1:]))

# roubo com concurso de pessoas (1/3 a 1/2) + arma de fogo (2/3), ambas da Parte Especial,
# e tentativa (Parte Geral): o motor mostra as duas opções do art. 68, parágrafo único
concurso_de_pessoas = ModifyingCause(
    "concurso_de_pessoas", "CP.art157.§2.II", AUMENTO, ESPECIAL, Fraction(1, 3), Fraction(1, 2)
)
arma_de_fogo = ModifyingCause("arma_de_fogo", "CP.art157.§2-A.I", AUMENTO, ESPECIAL, Fraction(2, 3))
roubo = calcular_pena_definitiva(
    Penalty(1460), [concurso_de_pessoas, arma_de_fogo, tentativa()], Composition.CASCATA
)
assert roubo.aplicando_todas.pena_definitiva == Penalty(2162)  # 1460 * 4/3 * 5/3 * 2/3 = 2162,96
assert roubo.limitada_art68 is not None
assert roubo.limitada_art68.pena_definitiva == Penalty(1622)  # 1460 * 5/3 * 2/3 = 1622,2 (prevalece a arma)
assert any(
    p.regra.startswith("art. 68, parágrafo único") and "concurso_de_pessoas" in p.motivo
    for p in roubo.limitada_art68.passos
)
assert not any("concurso_de_pessoas" in p.motivo for p in roubo.limitada_art68.passos if p.regra == "art. 68 do CP")

# concurso de diminuições da Parte Especial: prevalece a que mais diminui
dim_menor = ModifyingCause("dim_menor", "teste.a", DIMINUICAO, ESPECIAL, Fraction(1, 6))
dim_maior = ModifyingCause("dim_maior", "teste.b", DIMINUICAO, ESPECIAL, Fraction(1, 3))
diminuicoes = calcular_pena_definitiva(Penalty(360), [dim_menor, dim_maior], Composition.CASCATA)
assert diminuicoes.aplicando_todas.pena_definitiva == Penalty(200)  # 360 * 5/6 * 2/3
assert diminuicoes.limitada_art68.pena_definitiva == Penalty(240)  # 360 * 2/3

# diminuições que somam mais de 100% sobre a pena intermediária são rejeitadas (na cascata, não)
dois_tercos = ModifyingCause("dim_2_3", "teste.c", DIMINUICAO, GERAL, Fraction(2, 3))
try:
    calcular_pena_definitiva(Penalty(360), [dois_tercos, dois_tercos], Composition.SOBRE_PENA_INTERMEDIARIA)
    raise AssertionError("deveria ter rejeitado pena negativa")
except ValueError:
    pass
assert calcular_pena_definitiva(Penalty(360), [dois_tercos, dois_tercos], Composition.CASCATA).aplicando_todas.pena_definitiva == Penalty(40)

print("Fase 3 (pena definitiva): OK")

Fase 3 (pena definitiva): OK


## Dosimetria completa (as três fases encadeadas + alertas)

In [9]:
from dosimetria import SentencingResult, calcular_dosimetria_completa

faixa_roubo = PenaltyRange(
    minimo=Penalty.de_anos_meses_dias(anos=4),
    maximo=Penalty.de_anos_meses_dias(anos=10),
    origem="CP.art157",
)

# roubo: culpabilidade desfavorável (1/8 do intervalo de 2190 dias = 273), confissão,
# concurso de pessoas + arma de fogo (concurso de causas da Parte Especial)
roubo_completo = calcular_dosimetria_completa(
    faixa_roubo,
    com([JudicialCircumstance.CULPABILIDADE]),
    [atenuante()],
    [concurso_de_pessoas, arma_de_fogo],
    IntervalFraction(),
    Composition.CASCATA,
)
assert isinstance(roubo_completo, SentencingResult)
assert roubo_completo.faixa_aplicada == faixa_roubo
assert roubo_completo.pena_base == Penalty(1460 + 273)
assert roubo_completo.pena_intermediaria == Penalty(1460)
assert roubo_completo.pena_definitiva == Penalty(3244)  # 1460 * 4/3 * 5/3 = 3244,4
assert roubo_completo.alternativa_art68.pena_definitiva == Penalty(2433)  # 1460 * 5/3 = 2433,3
assert roubo_completo.criterio_quantum == IntervalFraction().nome
assert roubo_completo.composicao is Composition.CASCATA

# passo a passo completo e encadeado: mínimo da faixa -> pena-base -> intermediária -> definitiva
passos = roubo_completo.passos
assert [p.fase for p in passos] == [
    "1ª fase (pena-base)",
    "2ª fase (pena intermediária)",
    "3ª fase (pena definitiva)",
    "3ª fase (pena definitiva)",
]
assert passos[0].valor_antes == faixa_roubo.minimo
assert passos[-1].valor_depois == roubo_completo.pena_definitiva
assert all(a.valor_depois == b.valor_antes for a, b in zip(passos, passos[1:]))
assert any("art. 68, parágrafo único" in alerta for alerta in roubo_completo.alertas)

# furto sem nada desfavorável + confissão: Súmula 231 trava no mínimo e vira alerta
furto_231 = calcular_dosimetria_completa(
    faixa_furto, TODAS_NEUTRAS, [atenuante()], [], IntervalFraction(), Composition.CASCATA
)
assert furto_231.pena_definitiva == faixa_furto.minimo
assert furto_231.alternativa_art68 is None
assert len(furto_231.alertas) == 1 and "Súmula 231" in furto_231.alertas[0]

# furto com as 8 desfavoráveis + reincidência + repouso noturno:
# a 2ª fase trava no máximo, e a 3ª fase passa dele (as duas coisas viram alerta)
furto_maximo = calcular_dosimetria_completa(
    faixa_furto, todas_desfavoraveis, [agravante()], [repouso_noturno], IntervalFraction(), Composition.CASCATA
)
assert furto_maximo.pena_intermediaria == faixa_furto.maximo
assert furto_maximo.pena_definitiva == Penalty(1946)
assert any("máximo da faixa" in alerta and "agravante" in alerta for alerta in furto_maximo.alertas)
assert any("acima do máximo" in alerta for alerta in furto_maximo.alertas)

# quantum de 1/4 do intervalo com as 8 desfavoráveis estoura a faixa na 1ª fase -> alerta
furto_quantum_alto = calcular_dosimetria_completa(
    faixa_furto, todas_desfavoraveis, [], [], IntervalFraction(Fraction(1, 4)), Composition.CASCATA
)
assert furto_quantum_alto.pena_base == faixa_furto.maximo
assert any(alerta.startswith("pena-base calculada") for alerta in furto_quantum_alto.alertas)

print("Dosimetria completa: OK")

Dosimetria completa: OK


## Fundamentação (texto da dosimetria para leitura humana)

In [10]:
from dosimetria import gerar_fundamentacao

# roubo com duas majorantes da Parte Especial: todas as seções aparecem
texto = gerar_fundamentacao(roubo_completo)
print(texto)

assert texto.startswith("DOSIMETRIA DA PENA")
assert f"Faixa aplicada (CP.art157): de {faixa_roubo.minimo} a {faixa_roubo.maximo}." in texto
assert roubo_completo.criterio_quantum in texto
assert f"Pena-base: {roubo_completo.pena_base}." in texto
assert f"Pena intermediária: {roubo_completo.pena_intermediaria}." in texto
assert f"Pena definitiva: {roubo_completo.pena_definitiva}." in texto
assert f"Pena definitiva nesta opção: {roubo_completo.alternativa_art68.pena_definitiva}." in texto
for alerta in roubo_completo.alertas:
    assert f"- {alerta}" in texto

# as fases aparecem na ordem, e cada passo do resultado vira uma linha do texto
posicoes = [texto.index(titulo) for titulo in ["1ª FASE", "2ª FASE", "3ª FASE", "OPÇÃO DO ART. 68", "ALERTAS"]]
assert posicoes == sorted(posicoes)
for passo in roubo_completo.passos:
    assert f"- {passo.motivo} [{passo.regra}; {passo.dispositivo}]: {passo.valor_antes} -> {passo.valor_depois}" in texto

# os motivos das fases 1 e 2 nomeiam as circunstâncias
assert "(culpabilidade)" in roubo_completo.passos[0].motivo
assert "(confissao_espontanea)" in roubo_completo.passos[1].motivo

# sem alternativa nem alertas, essas seções não aparecem
texto_simples = gerar_fundamentacao(
    calcular_dosimetria_completa(faixa_furto, TODAS_NEUTRAS, [], [], IntervalFraction(), Composition.CASCATA)
)
assert "OPÇÃO DO ART. 68" not in texto_simples
assert "ALERTAS" not in texto_simples
assert "nenhuma circunstância desfavorável" in texto_simples

print()
print("Fundamentação: OK")

DOSIMETRIA DA PENA

Faixa aplicada (CP.art157): de 4 anos a 10 anos.
Critério de quantum nas fases 1 e 2: fração do intervalo (1/8).
Composição da 3ª fase: causas compostas em cascata (cada fração sobre a pena já modificada).

1ª FASE (PENA-BASE)
- 1 circunstância(s) desfavorável(is) do art. 59 (culpabilidade), fração do intervalo (1/8) cada [art. 59 do CP; CP.art157]: 4 anos -> 4 anos, 9 meses, 3 dias
Pena-base: 4 anos, 9 meses, 3 dias.

2ª FASE (PENA INTERMEDIÁRIA)
- 0 agravante(s), 1 atenuante(s) (confissao_espontanea) [arts. 61 a 67 do CP; CP.art157]: 4 anos, 9 meses, 3 dias -> 4 anos
Pena intermediária: 4 anos.

3ª FASE (PENA DEFINITIVA)
- concurso_de_pessoas: aumento de 1/3 (em cascata) [art. 68 do CP; CP.art157.§2.II]: 4 anos -> 5 anos, 4 meses, 1 dia
- arma_de_fogo: aumento de 2/3 (em cascata) [art. 68 do CP; CP.art157.§2-A.I]: 5 anos, 4 meses, 1 dia -> 8 anos, 10 meses, 24 dias
Pena definitiva: 8 anos, 10 meses, 24 dias.

OPÇÃO DO ART. 68, PARÁGRAFO ÚNICO, DO CP
(art. 68, pará

## Casos de dosimetria (dados/casos/dosimetrias.json)

In [11]:
import json

from dosimetria import entrada_de_dict

PASTA_CASOS = next(
    p / "dados" / "casos" for p in [Path.cwd(), *Path.cwd().parents] if (p / "dados" / "casos").is_dir()
)


def rodar_caso(caso):
    """Converte a entrada do caso (mesmo formato da API) e roda o motor."""
    return entrada_de_dict(caso["entrada"]).calcular()


casos = json.loads((PASTA_CASOS / "dosimetrias.json").read_text(encoding="utf-8"))["casos"]
assert len(casos) >= 10, "o plano pede ao menos 10 dosimetrias no conjunto de teste"

falhas = []
for caso in casos:
    esperado = caso["esperado"]
    resultado = rodar_caso(caso)
    obtido = {
        "pena_base_dias": resultado.pena_base.dias,
        "pena_intermediaria_dias": resultado.pena_intermediaria.dias,
        "pena_definitiva_dias": resultado.pena_definitiva.dias,
        "alternativa_art68_dias": resultado.alternativa_art68 and resultado.alternativa_art68.pena_definitiva.dias,
    }
    for chave, valor in obtido.items():
        if valor != esperado[chave]:
            falhas.append(f"{caso['id']}: {chave} esperado {esperado[chave]}, obtido {valor}")
    for trecho in esperado["alertas_contem"]:
        if not any(trecho in alerta for alerta in resultado.alertas):
            falhas.append(f"{caso['id']}: nenhum alerta contém {trecho!r} (alertas: {resultado.alertas})")
    if not esperado["alertas_contem"] and resultado.alertas:
        falhas.append(f"{caso['id']}: alertas inesperados {resultado.alertas}")
    # o passo a passo sempre sai do mínimo da faixa e termina na pena definitiva
    assert resultado.passos[0].valor_antes == resultado.faixa_aplicada.minimo
    assert resultado.passos[-1].valor_depois == resultado.pena_definitiva
    print(f"{caso['id']:<16} {str(resultado.pena_definitiva):<28} {caso['descricao'][:60]}")

assert not falhas, "\n".join(falhas)

# o índice do conjunto de treinamento aponta para casos que existem no arquivo de dosimetrias
treinamento = json.loads((PASTA_CASOS / "conjunto_treinamento.json").read_text(encoding="utf-8"))["casos"]
ids = {caso["id"] for caso in casos}
assert len(treinamento) == 10
for sentenca in treinamento:
    assert sentenca["tem_dosimetria"] == (sentenca["caso_dosimetria"] is not None)
    assert sentenca["caso_dosimetria"] is None or sentenca["caso_dosimetria"] in ids

print(f"Casos de dosimetria ({len(casos)}): OK")

treinamento-04   2 anos                       Furto qualificado por concurso de pessoas; pena-base no míni
construido-01    2 anos, 3 meses, 29 dias     Furto simples em repouso noturno, réu reincidente, culpabili
construido-02    8 anos                       Roubo com concurso de pessoas e arma de fogo, consequências 
construido-03    4 anos, 9 meses, 3 dias      Homicídio simples tentado; reincidência e confissão igualmen
construido-04    1 ano, 8 meses, 3 dias       Tráfico privilegiado: confissão sem efeito pela Súmula 231, 
construido-05    1 ano, 11 meses, 31 dias     Estelionato contra entidade de direito público, três circuns
construido-06    1 ano, 9 meses, 2 dias       Receptação: reincidência preponderante contra atenuante inom
construido-07    4 anos, 2 meses              Lesão corporal grave: 8 circunstâncias desfavoráveis, agrava
construido-08    2 anos, 4 meses, 1 dia       Furto qualificado em concurso formal (art. 70, Parte Geral),
construido-09    6 anos              

## Sentenças do conjunto de treinamento (lidas do PDF)

In [12]:
from dataclasses import replace

from sentencas import eh_penal, extrair_dosimetria_declarada, ler_sentencas

RAIZ = PASTA_CASOS.parent.parent
sentencas = ler_sentencas(RAIZ / "Conjunto de Treinamento - 10 Sentenças Judiciais.pdf")
anotacao = json.loads((PASTA_CASOS / "conjunto_treinamento.json").read_text(encoding="utf-8"))["casos"]
casos_por_id = {caso["id"]: caso for caso in casos}

# o leitor separa as 10 sentenças do PDF, e cada uma bate com a anotação feita à mão
assert [s.numero for s in sentencas] == list(range(1, 11))
for sentenca, esperado in zip(sentencas, anotacao):
    assert sentenca.numero == esperado["numero"]
    assert (sentenca.ramo, sentenca.tema, sentenca.processo, sentenca.resultado) == (
        esperado["ramo"], esperado["tema"], esperado["processo"], esperado["resultado"]
    ), f"caso {sentenca.numero}"
    assert sentenca.relatorio.startswith("Vistos.")
    assert sentenca.partes and sentenca.magistrado and sentenca.cargo
    assert eh_penal(sentenca) == esperado["tem_dosimetria"]

    declarada = extrair_dosimetria_declarada(sentenca)
    if not esperado["tem_dosimetria"]:
        # as 9 sentenças não penais: nenhuma dosimetria a extrair nem a calcular
        assert declarada is None, f"caso {sentenca.numero}"
        print(f"caso {sentenca.numero:>2}: {sentenca.ramo:<26} sem dosimetria (não penal)")
        continue

    # a dosimetria que o juiz escreveu, lida do PDF
    anotada = esperado["dosimetria_declarada"]
    assert declarada.pena_base == Penalty(anotada["pena_base_dias"])
    assert declarada.pena_definitiva == Penalty(anotada["pena_definitiva_dias"])
    assert declarada.dias_multa == anotada["dias_multa"]
    assert declarada.regime_inicial == anotada["regime_inicial"]

    # o motor, com os fatos do caso, chega à mesma pena que a sentença
    resultado = rodar_caso(casos_por_id[esperado["caso_dosimetria"]])
    assert resultado.pena_base == declarada.pena_base, (resultado.pena_base, declarada.pena_base)
    assert resultado.pena_definitiva == declarada.pena_definitiva, (resultado.pena_definitiva, declarada.pena_definitiva)
    print(
        f"caso {sentenca.numero:>2}: {sentenca.ramo:<26} sentença: {declarada.pena_definitiva}"
        f" | motor: {resultado.pena_definitiva}  -> iguais"
    )
    print()
    print("  Trecho da sentença:", declarada.trecho[: declarada.trecho.index("Substituo")].strip())
    print()
    print("  " + gerar_fundamentacao(resultado).replace("\n", "\n  "))
    print()

# o extrator de pena também entende anos, meses e dias por extenso (formato usual de sentença)
sentenca_penal = next(s for s in sentencas if eh_penal(s))
variante = replace(
    sentenca_penal,
    dispositivo=(
        "Dosimetria: Fixo a pena-base em 4 (quatro) anos e 6 (seis) meses de reclusão. "
        "Pena definitiva em 5 (cinco) anos, 3 (três) meses e 10 (dez) dias de reclusão "
        "e 12 (doze) dias-multa. Regime inicial Semiaberto."
    ),
)
declarada = extrair_dosimetria_declarada(variante)
assert declarada.pena_base == Penalty.de_anos_meses_dias(anos=4, meses=6)
assert declarada.pena_definitiva == Penalty.de_anos_meses_dias(anos=5, meses=3, dias=10)
assert declarada.dias_multa == 12
assert declarada.regime_inicial == "semiaberto"

print("Sentenças do conjunto de treinamento: OK")

caso  1: Direito do Consumidor      sem dosimetria (não penal)
caso  2: Direito de Família         sem dosimetria (não penal)
caso  3: Direito do Trabalho        sem dosimetria (não penal)
caso  4: Direito Penal              sentença: 2 anos | motor: 2 anos  -> iguais

  Trecho da sentença: Dosimetria: Fixada a pena-base no mínimo legal em 2 anos de reclusão. Ausentes agravantes ou atenuantes, bem como causas de aumento/diminuição. Pena definitiva em 2 (dois) anos de reclusão e 10 dias-multa.

  DOSIMETRIA DA PENA
  
  Faixa aplicada (CP.art155.§4.IV): de 2 anos a 8 anos.
  Critério de quantum nas fases 1 e 2: fração do intervalo (1/8).
  Composição da 3ª fase: causas compostas em cascata (cada fração sobre a pena já modificada).
  
  1ª FASE (PENA-BASE)
  - nenhuma circunstância desfavorável: pena-base fixada no mínimo da faixa [art. 59 do CP; CP.art155.§4.IV]: 2 anos -> 2 anos
  Pena-base: 2 anos.
  
  2ª FASE (PENA INTERMEDIÁRIA)
  - 0 agravante(s), 0 atenuante(s) [arts. 61 a 67 do 

## Entrada e saída em JSON (formato da API)

In [13]:
from copy import deepcopy

from dosimetria import resultado_para_dict

entrada_exemplo = casos_por_id["construido-02"]["entrada"]

# o JSON de saída traz as penas em dias e em texto, os passos, os alertas e a fundamentação
saida = resultado_para_dict(entrada_de_dict(entrada_exemplo).calcular())
assert saida["pena_definitiva"] == {"total_dias": 2920, "anos": 8, "meses": 0, "dias": 0, "texto": "8 anos"}
assert saida["alternativa_art68"]["pena_definitiva"]["total_dias"] == 2433
assert saida["composicao"] == "sobre_pena_intermediaria"
assert saida["fundamentacao"].startswith("DOSIMETRIA DA PENA")
assert [p["fase"] for p in saida["passos"]][:2] == ["1ª fase (pena-base)", "2ª fase (pena intermediária)"]
json.dumps(saida)  # serializável sem conversão extra


def erro_de(alterar):
    """Aplica uma alteração numa cópia da entrada de exemplo e devolve a mensagem de erro."""
    entrada = deepcopy(entrada_exemplo)
    alterar(entrada)
    try:
        entrada_de_dict(entrada)
    except ValueError as erro:
        return str(erro)
    raise AssertionError("deveria ter rejeitado a entrada")


# erros de formato viram mensagens que dizem o campo e as opções válidas
assert "circunstancias_desfavoraveis inválido: 'culpa'" in erro_de(
    lambda e: e.update(circunstancias_desfavoraveis=["culpa"])
)
assert "opções: culpabilidade, antecedentes" in erro_de(lambda e: e.update(circunstancias_desfavoraveis=["culpa"]))
assert "composicao inválido" in erro_de(lambda e: e.update(composicao="soma"))
assert "estrategia.tipo inválido" in erro_de(lambda e: e["estrategia"].update(tipo="metade"))
assert "fração inválida: '1,3'" in erro_de(lambda e: e["causas"][0].update(fracao_min="1,3"))
assert "campo obrigatório ausente em entrada: 'faixa'" in erro_de(lambda e: e.pop("faixa"))
assert "campos desconhecidos" in erro_de(lambda e: e["faixa"].update(minimo={"semanas": 3}))
assert "valores negativos" in erro_de(lambda e: e["faixa"].update(minimo={"anos": -1}))
# regras do motor continuam valendo: fração acima da mínima sem justificativa
assert "exige justificativa" in erro_de(lambda e: e["causas"][0].update(fracao_escolhida="1/2"))

print("Entrada e saída em JSON: OK")

Entrada e saída em JSON: OK


## API (FastAPI)

In [14]:
from fastapi.testclient import TestClient

from api.app import app, limite_calculo
from api.esquemas import EXEMPLO_ENTRADA, ComparisonRequest

cliente = TestClient(app)
limite_calculo.limpar()

# rotas gerais e documentação interativa
assert cliente.get("/saude").json()["status"] == "ok"
assert cliente.get("/docs").status_code == 200
assert cliente.get("/openapi.json").json()["info"]["title"] == "sergius-ia-Judge"
assert cliente.get("/opcoes").json()["composicoes"] == ["cascata", "sobre_pena_intermediaria"]

# os exemplos da API são os mesmos casos testados acima, e cada um dá o resultado esperado
exemplos = cliente.get("/exemplos").json()
assert [e["id"] for e in exemplos] == [c["id"] for c in casos]
for caso in casos:
    entrada = cliente.get(f"/exemplos/{caso['id']}").json()["entrada"]
    resposta = cliente.post("/dosimetria/calcular", json=entrada)
    assert resposta.status_code == 200, (caso["id"], resposta.json())
    corpo = resposta.json()
    assert corpo["pena_definitiva"]["total_dias"] == caso["esperado"]["pena_definitiva_dias"], caso["id"]
    alternativa = corpo["alternativa_art68"]
    assert (alternativa and alternativa["pena_definitiva"]["total_dias"]) == caso["esperado"]["alternativa_art68_dias"]
assert cliente.get("/exemplos/nao-existe").status_code == 404

# o exemplo mostrado na documentação funciona
corpo = cliente.post("/dosimetria/calcular", json=EXEMPLO_ENTRADA).json()
assert corpo["pena_definitiva"]["texto"] == "2 anos, 3 meses, 29 dias"
assert corpo["fundamentacao"].startswith("DOSIMETRIA DA PENA")

# correção do estudante: acerta as fases 1 e 2, erra a 3ª por 1 dia
pedido = ComparisonRequest.model_config["json_schema_extra"]["examples"][0]
correcao = cliente.post("/ensino/comparar", json=pedido).json()
assert (correcao["acertos"], correcao["total"]) == (2, 3)
assert [f["correta"] for f in correcao["fases"]] == [True, True, False]
assert correcao["fases"][2]["diferenca_dias"] == 1
assert correcao["fases"][2]["explicacao"] == ["repouso_noturno: aumento de 1/3 (em cascata)"]

# na pena definitiva, a opção do art. 68, parágrafo único, também conta como certa
roubo = cliente.get("/exemplos/construido-02").json()["entrada"]
opcao_limitada = {"pena_definitiva": {"anos": 6, "meses": 8, "dias": 3}}  # 2433 dias
correcao = cliente.post("/ensino/comparar", json={"entrada": roubo, "resposta": opcao_limitada}).json()
assert correcao["fases"][0]["correta"] and "art. 68, parágrafo único" in correcao["fases"][0]["observacao"]
assert cliente.post("/ensino/comparar", json={"entrada": roubo, "resposta": {}}).status_code == 422

# erros de formato: 422 com o campo e a mensagem em português
entrada_ruim = json.loads(json.dumps(roubo))
entrada_ruim["composicao"] = "soma"
entrada_ruim["causas"][0]["fracao_min"] = "1,3"
del entrada_ruim["estrategia"]
erros = {e["campo"]: e["mensagem"] for e in cliente.post("/dosimetria/calcular", json=entrada_ruim).json()["detail"]}
assert erros == {
    "composicao": "valor inválido (opções: 'cascata', 'sobre_pena_intermediaria')",
    "causas.0.fracao_min": "formato inválido (frações no formato '1/3')",
    "estrategia": "campo obrigatório ausente",
}

# regras do motor também voltam como 422 com a explicação
sem_justificativa = json.loads(json.dumps(roubo))
sem_justificativa["causas"][0]["fracao_escolhida"] = "1/2"
resposta = cliente.post("/dosimetria/calcular", json=sem_justificativa)
assert resposta.status_code == 422
assert resposta.json()["detail"] == "concurso_de_pessoas: fração 1/2 acima da mínima exige justificativa"

# limites de entrada: valores e listas fora do razoável são recusados antes do cálculo
enorme = json.loads(json.dumps(EXEMPLO_ENTRADA))
enorme["faixa"]["maximo"] = {"anos": 1001}
enorme["causas"][0]["fracao_min"] = "1/99999"
enorme["agravantes_atenuantes"] = enorme["agravantes_atenuantes"] * 31
erros = {e["campo"]: e["mensagem"] for e in cliente.post("/dosimetria/calcular", json=enorme).json()["detail"]}
assert erros == {
    "faixa.maximo.anos": "não pode passar de 1000",
    "causas.0.fracao_min": "formato inválido (frações no formato '1/3')",
    "agravantes_atenuantes": "pode ter no máximo 30 itens",
}, erros

# limite de requisições do cálculo e da correção: 60 por minuto por visitante, no mesmo balde
limite_calculo.limpar()
codigos = [cliente.post("/dosimetria/calcular", json=EXEMPLO_ENTRADA).status_code for _ in range(61)]
assert codigos[:60] == [200] * 60 and codigos[60] == 429
assert cliente.post("/ensino/comparar", json=pedido).status_code == 429
limite_calculo.limpar()

print("API: OK")

API: OK


## Páginas para estudantes (web/)

In [15]:
import re

# as quatro páginas respondem em HTML, com o menu marcando a página atual
paginas = {
    "/": "Início",
    "/calcular": "Calcular",
    "/praticar": "Praticar",
    "/pesquisar": "Pesquisar na lei",
    "/como-funciona": "Como funciona",
}
for rota, titulo in paginas.items():
    resposta = cliente.get(rota)
    assert resposta.status_code == 200, rota
    assert resposta.headers["content-type"].startswith("text/html"), rota
    html = resposta.text
    assert f"<title>{titulo} · sergius-ia-Judge</title>" in html, rota
    assert re.search(rf'<a href="{re.escape(rota)}"\s+aria-current="page">', html), f"menu não marca {rota}"
    # cada página referencia só arquivos estáticos que existem
    for arquivo in re.findall(r'(?:href|src)="(/static/[^"?]+)', html):
        assert cliente.get(arquivo).status_code == 200, f"{rota} referencia {arquivo}, que não existe"

# o JavaScript das páginas chama só rotas que a API tem
rotas_api = set(cliente.get("/openapi.json").json()["paths"])
for arquivo in ["comum.js", "calcular.js", "praticar.js", "pesquisar.js", "agente.js", "analisar.js"]:
    codigo = cliente.get(f"/static/{arquivo}").text
    for chamada in re.findall(r'chamarApi\(\s*[`"](/[a-z/]+)', codigo):
        assert chamada in rotas_api or any(r.startswith(chamada.rstrip('/') + "/{") for r in rotas_api), f"{arquivo} chama {chamada}"

# os IDs usados pelo JavaScript existem no HTML de cada página
calcular_html = cliente.get("/calcular").text
for id_ in ["formulario", "exemplo", "limpar", "circunstancias", "agravantes", "causas", "erros", "resultado", "modelo-agravante", "modelo-causa"]:
    assert f'id="{id_}"' in calcular_html, id_
praticar_html = cliente.get("/praticar").text
for id_ in ["caso", "sortear", "enunciado", "resposta", "erros", "correcao"]:
    assert f'id="{id_}"' in praticar_html, id_
pesquisar_html = cliente.get("/pesquisar").text
for id_ in ["form-busca", "q", "lei", "erros", "avisos", "resultados"]:
    assert f'id="{id_}"' in pesquisar_html, id_
# "origem" é o nome de um campo das causas; o campo da faixa precisa de nome próprio (teste de regressão)
assert 'name="faixa_origem"' in calcular_html

# a API em JSON continua fora das páginas: /docs e as rotas antigas seguem iguais
assert cliente.get("/docs").status_code == 200
assert {"/dosimetria/calcular", "/ensino/comparar", "/exemplos", "/opcoes", "/saude"} <= rotas_api
assert not any(rota in rotas_api for rota in paginas), "as páginas não devem aparecer na documentação da API"

print("Páginas: OK")

Páginas: OK


## Anonimização (privacidade/)

In [16]:
from privacidade import anonimizar


def anon(texto):
    return anonimizar(texto).texto


# documentos, contatos e identificadores
resultado = anonimizar(
    "CPF 123.456.789-09, CNPJ 12.345.678/0001-90, RG 12.345.678-9, e-mail fulano@exemplo.com.br, "
    "telefone (21) 98765-4321, CEP 20000-000, placa ABC1D23, processo 0089123-11.2023.8.19.0001, "
    "morador da Rua das Flores, 120."
)
assert resultado.contagem_por_tipo() == {
    "PROCESSO": 1, "CNPJ": 1, "CPF": 1, "RG": 1, "EMAIL": 1, "TELEFONE": 1, "CEP": 1, "PLACA": 1, "ENDERECO": 1
}, resultado.contagem_por_tipo()
for dado in ["123.456", "12.345.678", "fulano", "98765", "20000", "ABC1D23", "0089123", "Flores"]:
    assert dado not in resultado.texto, dado

# nomes: compostos, em maiúsculas, após papel processual, prenome solto, menção repetida
assert anon("O réu Rafael Souza e Silva furtou. Rafael fugiu.") == "O réu [PESSOA 1] furtou. [PESSOA 1] fugiu."
assert anon("RAFAEL SOUZA E SILVA foi denunciado.") == "[PESSOA 1] foi denunciado."
assert anon("A vítima Ana contou tudo.") == "A vítima [PESSOA 1] contou tudo."
assert anon("Levou os bens de Maria. Depois, Maria gritou.") == "Levou os bens de [PESSOA 1]. Depois, [PESSOA 1] gritou."
assert anon("No dia 3, Rafael Souza entrou. Segundo a testemunha, Rafael fugiu.") == (
    "No dia 3, [PESSOA 1] entrou. Segundo a testemunha, [PESSOA 1] fugiu."
)
# pessoas diferentes recebem marcadores diferentes
assert anon("Paulo e a esposa Carla foram assaltados.") == "[PESSOA 1] e a esposa [PESSOA 2] foram assaltados."

# instituições, termos jurídicos e lugares não são pessoas
preservado = (
    "O Ministério Público denunciou com base no art. 155 do Código Penal. A Defensoria Pública recorreu ao "
    "Superior Tribunal de Justiça, que aplicou a Súmula 231. O fato ocorreu em São Paulo, depois em João Pessoa "
    "e em Mato Grosso do Sul."
)
assert anon(preservado) == preservado, anon(preservado)
assert anon("Em Vitória, Paulo reagiu.") == "Em Vitória, [PESSOA 1] reagiu."

# apelidos, data de nascimento e bairro; a idade fica, porque muda a pena (arts. 61, II, h, e 65, I)
assert anon(
    'O réu Rafael Souza, vulgo "Baixinho", nascido em 12/03/1990, morador do bairro Jardim América, tinha 34 anos.'
) == "O réu [PESSOA 1], vulgo [APELIDO], nascido em [NASCIMENTO], morador do bairro [BAIRRO], tinha 34 anos."
assert anon("O acusado, vulgo baixinho, foi preso.") == "O acusado, vulgo [APELIDO], foi preso."
assert anon("Data de nascimento: 5 de maio de 2001.") == "Data de nascimento: [NASCIMENTO]."
assert anon("O réu é conhecido como autor de furtos.") == "O réu é conhecido como autor de furtos."

# nome depois de "identificada como", "de nome": também é trocado
assert anon("a vítima foi uma mulher de 29 anos identificada como Lane") == "a vítima foi uma mulher de 29 anos identificada como [PESSOA 1]"
assert anon("um rapaz de nome Kauê fugiu") == "um rapaz de nome [PESSOA 1] fugiu"

# a sentença penal do conjunto de treinamento sai sem o nome do réu
sentenca_penal = next(s for s in sentencas if eh_penal(s))
texto_sentenca = anon(sentenca_penal.relatorio + " " + sentenca_penal.dispositivo)
assert "RAFAEL" not in texto_sentenca.upper().replace("[PESSOA", "")
assert "MINISTÉRIO PÚBLICO" in texto_sentenca and "Defensoria Pública" in texto_sentenca

print("Anonimização: OK")

Anonimização: OK


## Casos e segurança (api/casos.py, api/seguranca.py, banco/)

In [17]:
import os

from sqlalchemy import text

from api.casos import limite_consulta, limite_exclusao, limite_previa, limite_salvar
from banco import banco_configurado, obter_engine
from banco.migrar import migrar

limites = [limite_previa, limite_salvar, limite_consulta, limite_exclusao]
for limite in limites:
    limite.limpar()

# ---------- segurança que não depende do banco ----------
cabecalhos = cliente.get("/calcular").headers
assert "script-src 'self'" in cabecalhos["content-security-policy"]
assert "frame-ancestors 'none'" in cabecalhos["content-security-policy"]
assert cabecalhos["x-frame-options"] == "DENY"
assert cabecalhos["x-content-type-options"] == "nosniff"
assert cabecalhos["referrer-policy"] == "strict-origin-when-cross-origin"
assert "server" not in {k.lower() for k in cabecalhos} or "uvicorn" not in cabecalhos.get("server", "")

# corpo grande demais: 413, com ou sem Content-Length
gigante = b'{"descricao":"' + b"a" * 300_000 + b'"}'
assert cliente.post("/casos/previa", content=gigante, headers={"content-type": "application/json"}).status_code == 413
pedacos = iter([b'{"descricao":"'] + [b"a" * 10_000] * 40 + [b'"}'])
assert cliente.post("/casos/previa", content=pedacos, headers={"content-type": "application/json"}).status_code == 413

# validação de tamanho, sem ecoar o texto inteiro
erro = cliente.post("/casos/previa", json={"descricao": "x" * 20_001}).json()["detail"][0]
assert erro["mensagem"] == "pode ter no máximo 20000 caracteres" and len(erro["valor_recebido"]) <= 61
erro = cliente.post("/casos/previa", json={"descricao": "curto"}).json()["detail"][0]
assert erro["mensagem"] == "precisa ter pelo menos 50 caracteres"

# outro site não pode enviar casos pelo navegador de um visitante (CORS só libera GET)
preflight = cliente.options(
    "/casos", headers={"origin": "https://site-malicioso.example", "access-control-request-method": "POST"}
)
assert preflight.status_code == 400

# prévia: anonimiza sem gravar
DESCRICAO = (
    "O réu Rafael Souza, reincidente, CPF 123.456.789-09, entrou à noite na loja da Rua das Flores, 120, "
    "e subtraiu o celular da vítima Maria, de 72 anos. Rafael confessou."
)
previa = cliente.post("/casos/previa", json={"descricao": DESCRICAO}).json()
assert "Rafael" not in previa["descricao_anonimizada"] and "123.456" not in previa["descricao_anonimizada"]
assert previa["contagem"] == {"CPF": 1, "ENDERECO": 1, "PESSOA": 3}

# limite de requisições: por cliente, e sem aceitar IP falsificado no X-Forwarded-For
limite_previa.limpar()
codigos = [cliente.post("/casos/previa", json={"descricao": DESCRICAO}).status_code for _ in range(31)]
assert codigos[:30] == [200] * 30 and codigos[30] == 429
os.environ["CONFIAR_PROXY"] = "1"
try:
    limite_previa.limpar()
    falsificados = [
        cliente.post(
            "/casos/previa", json={"descricao": DESCRICAO}, headers={"x-forwarded-for": f"10.0.0.{i}, 200.1.1.1"}
        ).status_code
        for i in range(31)
    ]
    assert falsificados[30] == 429, "IP falsificado à esquerda do X-Forwarded-For não pode burlar o limite"
    assert "strict-transport-security" in cliente.get("/", headers={"x-forwarded-proto": "https"}).headers
finally:
    del os.environ["CONFIAR_PROXY"]
    limite_previa.limpar()
assert "strict-transport-security" not in cliente.get("/", headers={"x-forwarded-proto": "https"}).headers

# teste de regressão: no Render, atrás do Cloudflare, o último valor do
# X-Forwarded-For é o IP do Cloudflare, que muda a cada requisição, e o limite ficava burlável.
# O IP confiável é o cf-connecting-ip, que o Cloudflare sobrescreve.
os.environ.update({"CONFIAR_PROXY": "1", "IP_CLIENTE_CABECALHO": "cf-connecting-ip"})
try:
    limite_previa.limpar()
    mesmo_visitante = [
        cliente.post(
            "/casos/previa",
            json={"descricao": DESCRICAO},
            headers={"x-forwarded-for": f"10.9.{i}.{i}, 172.68.{i}.1", "cf-connecting-ip": "200.1.1.1"},
        ).status_code
        for i in range(31)
    ]
    assert mesmo_visitante[30] == 429, "X-Forwarded-For variando não pode burlar o limite atrás do Cloudflare"
    limite_previa.limpar()
    visitantes = [
        cliente.post("/casos/previa", json={"descricao": DESCRICAO}, headers={"cf-connecting-ip": f"200.1.1.{i}"}).status_code
        for i in range(40)
    ]
    assert set(visitantes) == {200}, "visitantes diferentes não podem dividir o mesmo limite"
    limite_previa.limpar()
    sem_cabecalho = [
        cliente.post("/casos/previa", json={"descricao": DESCRICAO}, headers={"x-forwarded-for": f"10.0.0.{i}"}).status_code
        for i in range(31)
    ]
    assert sem_cabecalho[30] == 429, "sem o cabeçalho do proxy, não pode cair para valores forjáveis"
finally:
    for chave in ("CONFIAR_PROXY", "IP_CLIENTE_CABECALHO"):
        os.environ.pop(chave, None)
    limite_previa.limpar()

# no Render, a proteção vale mesmo que as variáveis do render.yaml não sejam aplicadas
# (a detecção não depende delas): RENDER=true basta para usar o cf-connecting-ip
os.environ["RENDER"] = "true"
try:
    limite_previa.limpar()
    so_render = [
        cliente.post(
            "/casos/previa",
            json={"descricao": DESCRICAO},
            headers={"x-forwarded-for": f"10.7.{i}.{i}, 172.68.{i}.1", "cf-connecting-ip": "200.2.2.2"},
        ).status_code
        for i in range(31)
    ]
    assert so_render[30] == 429, "no Render, o limite tem que usar o cf-connecting-ip automaticamente"
    saude = cliente.get("/saude", headers={"cf-connecting-ip": "200.2.2.2"}).json()["protecao_ip"]
    assert saude == {"origem": "cf-connecting-ip", "cabecalho_presente": True}, saude
finally:
    os.environ.pop("RENDER", None)
    limite_previa.limpar()
assert cliente.get("/saude").json()["protecao_ip"]["origem"] == "conexão direta"

# JSON ilegível: mensagem em português
ilegivel = cliente.post("/casos/previa", content="{inválido".encode("latin-1"), headers={"content-type": "application/json"})
assert ilegivel.status_code == 400 and "JSON válido" in ilegivel.json()["detail"]

# ---------- banco em produção: só com TLS verificado ----------
# sem o certificado CA, a conexão seria criptografada mas sem conferir o servidor: um
# intermediário receberia a senha e os casos. Em produção, o banco fica desligado.
from pathlib import Path

import certifi

from banco.config import bloqueio_de_seguranca, url_configurada

variaveis = ("RENDER", "EXIGIR_TLS_VERIFICADO", "MYSQL_CA_CERT", "DATABASE_URL", "DATABASE_URL_MIGRACAO")
salvas = {chave: os.environ.get(chave) for chave in variaveis}
try:
    os.environ.pop("MYSQL_CA_CERT", None)
    os.environ.setdefault("DATABASE_URL", "mysql://usuario@127.0.0.1:1/banco?ssl-mode=REQUIRED")
    assert bloqueio_de_seguranca() is None, "fora de produção, o banco local funciona sem certificado"
    os.environ["RENDER"] = "true"
    assert "MYSQL_CA_CERT não definido" in bloqueio_de_seguranca()
    assert not banco_configurado() and migrar() is False
    assert cliente.get("/saude").json()["banco"] == "desligado"
    limite_salvar.limpar()
    assert cliente.post("/casos", json={"descricao": DESCRICAO, "consentimento": True}).status_code == 503
    os.environ["MYSQL_CA_CERT"] = "não é um certificado"
    assert "MYSQL_CA_CERT inválido" in bloqueio_de_seguranca()
    os.environ["MYSQL_CA_CERT"] = Path(certifi.where()).read_text(encoding="utf-8")
    assert bloqueio_de_seguranca() is None and banco_configurado()
    os.environ.pop("RENDER")
    os.environ["EXIGIR_TLS_VERIFICADO"] = "1"
    os.environ.pop("MYSQL_CA_CERT")
    assert bloqueio_de_seguranca() is not None, "EXIGIR_TLS_VERIFICADO=1 também exige o certificado"

    # migrações com outro usuário (privilégio mínimo); sem ele, usam a DATABASE_URL
    os.environ.pop("DATABASE_URL_MIGRACAO", None)
    assert url_configurada(migracao=True) == url_configurada()
    os.environ["DATABASE_URL_MIGRACAO"] = "mysql://migracao@127.0.0.1:1/banco"
    assert url_configurada(migracao=True) == "mysql://migracao@127.0.0.1:1/banco"
    assert url_configurada() != url_configurada(migracao=True)
finally:
    for chave, valor in salvas.items():
        if valor is None:
            os.environ.pop(chave, None)
        else:
            os.environ[chave] = valor
    limite_salvar.limpar()
assert cliente.get("/saude").json()["banco"] == ("ligado" if banco_configurado() else "desligado")
# a API sobe sem a senha do usuário das migrações no ambiente
assert "env -u DATABASE_URL_MIGRACAO uvicorn" in (RAIZ / "Dockerfile").read_text(encoding="utf-8")

# ---------- casos no banco (MySQL) ----------
if not banco_configurado():
    assert os.environ.get("CI") != "true", "no CI, o MySQL é obrigatório (DATABASE_URL não definida)"
    print("Casos e segurança: OK (parte do banco PULADA: sem DATABASE_URL; rode python scripts/preparar_ambiente.py "
          "e docker compose up -d mysql)")
else:
    migrar()
    sem_consentimento = cliente.post("/casos", json={"descricao": DESCRICAO, "consentimento": False})
    assert sem_consentimento.status_code == 422

    salvo = cliente.post("/casos", json={"descricao": DESCRICAO, "consentimento": True})
    assert salvo.status_code == 201, salvo.text
    caso = salvo.json()
    assert caso["status"] == "recebido" and len(caso["codigo"]) >= 8
    assert salvo.headers["cache-control"] == "no-store"

    # o banco guarda só o texto anonimizado: nenhum dado original, em nenhuma coluna
    with obter_engine().connect() as conexao:
        linha = conexao.execute(text("SELECT * FROM casos WHERE codigo = :c"), {"c": caso["codigo"]}).mappings().one()
    gravado = " ".join(str(valor) for valor in linha.values())
    for dado in ["Rafael", "Souza", "123.456.789-09", "Flores", "Maria"]:
        assert dado not in gravado, f"dado pessoal gravado no banco: {dado}"

    consultado = cliente.get(f"/casos/{caso['codigo']}")
    assert consultado.status_code == 200 and consultado.json()["descricao"] == caso["descricao"]

    # códigos inválidos ou inexistentes: 404/422 sem vazar nada
    assert cliente.get("/casos/AAAAAAAAAAAA").status_code == 404
    assert cliente.get("/casos/curto").status_code == 422
    assert cliente.get("/casos/%27%20OR%201%3D1%20--").status_code == 422  # tentativa de SQL injection

    # exclusão (LGPD)
    assert cliente.delete(f"/casos/{caso['codigo']}").status_code == 204
    assert cliente.get(f"/casos/{caso['codigo']}").status_code == 404
    assert cliente.delete(f"/casos/{caso['codigo']}").status_code == 404

    # limite de envios: 5 a cada 10 minutos por cliente
    limite_salvar.limpar()
    envios = [cliente.post("/casos", json={"descricao": DESCRICAO, "consentimento": True}) for _ in range(6)]
    assert [e.status_code for e in envios] == [201] * 5 + [429]
    for envio in envios[:5]:
        cliente.delete(f"/casos/{envio.json()['codigo']}")

    # prazo de guarda (LGPD): recebidos e rejeitados com mais de 90 dias são apagados, e
    # validados ficam. A limpeza roda a cada envio e na inicialização (banco.migrar).
    from banco import apagar_casos_expirados, fabrica_de_sessoes

    limite_salvar.limpar()
    limite_exclusao.limpar()
    velhos = [cliente.post("/casos", json={"descricao": DESCRICAO, "consentimento": True}).json()["codigo"] for _ in range(3)]
    with obter_engine().begin() as conexao:
        conexao.execute(
            text("UPDATE casos SET criado_em = criado_em - INTERVAL 91 DAY WHERE codigo IN (:a, :b, :c)"),
            dict(zip("abc", velhos)),
        )
        conexao.execute(text("UPDATE casos SET status = 'rejeitado' WHERE codigo = :c"), {"c": velhos[1]})
        conexao.execute(text("UPDATE casos SET status = 'validado' WHERE codigo = :c"), {"c": velhos[2]})
    recente = cliente.post("/casos", json={"descricao": DESCRICAO, "consentimento": True}).json()["codigo"]
    assert [cliente.get(f"/casos/{c}").status_code for c in [*velhos, recente]] == [404, 404, 200, 200]
    with fabrica_de_sessoes()() as sessao:
        assert apagar_casos_expirados(sessao) == 0
    for codigo in (velhos[2], recente):
        assert cliente.delete(f"/casos/{codigo}").status_code == 204
    print("Casos e segurança: OK (com MySQL)")

for limite in limites:
    limite.limpar()

banco: migrações aplicadas


Casos e segurança: OK (com MySQL)


## Base de legislação e busca (fontes/, RAG)

Os artigos extraídos dos PDFs oficiais (`dados/fontes/dispositivos.json`, gerado por `scripts/construir_base_de_fontes.py`), a busca que o agente usa para achar a lei que responde a uma pergunta, e os casos de teste da busca (`dados/fontes/perguntas.json`).

In [ ]:
from collections import Counter

from api.app import limite_calculo
from api.fontes import limite_fontes
from fontes.avaliacao import avaliar
from fontes.busca import base_de_fontes, radical
from fontes.extracao import LegalProvision, cabecalho_de_lei, dividir_em_artigos, escolher_versoes
from fontes.texto import corrigir_ordinais, juntar_palavras_partidas, limpar_paginas

base = base_de_fontes()

# ---------- limpeza do texto que sai do PDF ----------
assert juntar_palavras_partidas("rompimento de obs-\ntáculo") == "rompimento de obstáculo"
assert juntar_palavras_partidas("considera-\nse crime") == "considera-se crime"
assert juntar_palavras_partidas("aplicar -\n-se-á") == "aplicar-se-á"
assert corrigir_ordinais("Art. 1 o Não há crime") == "Art. 1º Não há crime"
assert corrigir_ordinais("§ 4o-A. A pena") == "§ 4º-A. A pena"
assert corrigir_ordinais("Lei no 8.072") == "Lei nº 8.072"
# cabeçalho e número de página repetidos somem; nota de rodapé grudada no fim da linha também,
# mas o número de uma lei no fim da linha fica ("Lei nº 10.406")
paginas = [
    f"{36 + i}\nColetânea básica penal\nArt. {i + 1}. Texto do artigo, previsto na Lei nº 10.406\n"
    "Parágrafo único. Vale o disposto no crime:5"
    for i in range(6)
]
linhas = limpar_paginas(paginas)
assert "Coletânea básica penal" not in linhas and not any(linha.isdigit() for linha in linhas), linhas
assert linhas[:2] == ["Art. 1. Texto do artigo, previsto na Lei nº 10.406", "Parágrafo único. Vale o disposto no crime:"], linhas[:2]

# ---------- títulos de lei ----------
assert cabecalho_de_lei(["Lei nº 8.072/1990", "Dispõe sobre os crimes hediondos"], 0) == "Lei 8.072/1990"
assert cabecalho_de_lei(["LEI Nº 8.078, DE 11 DE SETEMBRO DE 1990", "O presidente da República"], 0) == "Lei 8.078/1990"
assert cabecalho_de_lei(["LEi No 10.406", "DE 10 DE JANEiro DE 2002", "Institui o Código Civil."], 0) == "Lei 10.406/2002"
# a mesma lei citada no meio de uma frase não é o começo de outra lei (teste de regressão)
assert cabecalho_de_lei(["as penas previstas no", "Decreto-lei nº 2.848, de 7 de dezembro de 1940", "(Código Penal), e no art. 1º"], 1) is None

# ---------- artigos: epígrafe, e artigo citado por lei que altera outra ----------
linhas_da_lei = [
    "Furto", "Art. 1º Subtrair coisa alheia móvel:", "Pena – reclusão, de um a quatro anos.",
    "Art. 2º O art. 172 do Código Penal passa a vigorar com a seguinte redação:",
    "Art. 172. Emitir fatura que não corresponda à mercadoria vendida.",
    "Roubo", "Art. 3º Subtrair mediante grave ameaça:", "Art. 4º Esta Lei entra em vigor na data de sua publicação.",
]
artigos = dividir_em_artigos("Lei 9.999/2099", "Lei de teste", linhas_da_lei, "teste")
assert [a.artigo for a in artigos] == ["1", "2", "3", "4"], [a.artigo for a in artigos]
assert (artigos[0].rotulo, artigos[0].epigrafe, artigos[2].epigrafe) == ("L9999.art1", "Furto", "Roubo")
assert "Art. 172. Emitir fatura" in artigos[1].texto, "artigo citado fica dentro do artigo que o cita"
assert artigos[0].texto == "Art. 1º Subtrair coisa alheia móvel:\nPena – reclusão, de um a quatro anos."

# ---------- versão de cada artigo: a edição mais nova vence, mesmo que traga só parte da lei ----------
def artigo_de_teste(numero, texto, fonte):
    return LegalProvision(f"L1.art{numero}", "L1", "Lei 1", "Lei 1/2000", str(numero), "", (), texto, False, fonte)

antiga = [artigo_de_teste(n, "texto completo do artigo " * 4, "antiga") for n in range(1, 11)]
nova = [artigo_de_teste(n, "texto novo e completo do artigo " * 4, "nova") for n in (6, 7)]
versoes = {a.rotulo: a.fonte for a in escolher_versoes({"antiga": antiga, "nova": nova}, {"antiga": "2008-07", "nova": "2026-01"})}
assert len(versoes) == 10 and versoes["L1.art1"] == "antiga" and versoes["L1.art6"] == versoes["L1.art7"] == "nova", versoes

# ---------- a base gerada dos PDFs ----------
por_lei = Counter(d.lei for d in base.dispositivos)
assert por_lei["CP"] >= 400 and por_lei["CPP"] >= 800 and por_lei["CF"] >= 250 and por_lei["LEP"] >= 200 and por_lei["CC"] >= 2000, por_lei
rotulos = [d.rotulo for d in base.dispositivos]
assert len(rotulos) == len(set(rotulos)), "rótulo repetido na base"
assert all(d.texto.startswith("Art.") for d in base.dispositivos)
# critério de pronto da fase 1 do plano
assert base.trecho_do_rotulo("CP.art155.§4.IV")[1] == "IV – mediante concurso de duas ou mais pessoas;"
furto = base.obter("CP.art155")
assert furto.epigrafe == "Furto" and "Pena – reclusão, de um a quatro anos, e multa." in furto.texto
assert "§ 2º-A. A pena aumenta-se de 2/3 (dois terços):" in base.obter("CP.art157").texto
# de cada lei, a edição mais recente: o CPP da Coletânea (2026) vence o da 5ª ed. (2023), e a parte
# criminal da Lei 9.099 da Coletânea vence a do livro do Código Civil (2008), que dá o resto da lei
assert base.obter("CPP.art1").fonte == "coletanea-penal-16"
assert base.obter("L9099.art89").fonte == "coletanea-penal-16" and base.obter("L9099.art3").fonte == "cc-2"

# ---------- verificador de citações: todo dispositivo dos casos de dosimetria existe na base ----------
rotulos_dos_casos = set()
for caso in json.loads((PASTA_CASOS / "dosimetrias.json").read_text(encoding="utf-8"))["casos"]:
    entrada = caso["entrada"]
    rotulos_dos_casos.add(entrada["faixa"]["origem"])
    rotulos_dos_casos.update(item["dispositivo"] for item in entrada["agravantes_atenuantes"] + entrada["causas"])
fora_da_base = sorted(r for r in rotulos_dos_casos if base.trecho_do_rotulo(r) is None)
# a Lei de Drogas não veio em nenhum PDF: fica registrada aqui até entrar na base
assert fora_da_base == ["L11343.art33", "L11343.art33.§4", "L11343.art40.VI", "L11343.art41"], fora_da_base
assert all(base.trecho_do_rotulo(r)[2] for r in rotulos_dos_casos - set(fora_da_base)), "parágrafo, inciso ou alínea não encontrado"
assert base.trecho_do_rotulo("CP.art61.II.h")[1].startswith("h) contra criança, maior de 60 (sessenta) anos")

# ---------- busca: casos de teste ----------
assert radical("furtou") == radical("furtada") == radical("furto")
resultados_busca, metricas = avaliar(base)
falhas = [(r.pergunta, r.posicao, r.ate, r.encontrados[:3]) for r in resultados_busca if not r.passou]
assert not falhas, falhas
assert metricas["acerto@5"] == 1.0 and metricas["acerto@1"] >= 0.85 and metricas["mrr"] >= 0.9, metricas
assert [r.dispositivo.rotulo for r in base.buscar("furto noturno")] == [r.dispositivo.rotulo for r in base.buscar("furto noturno")]
assert all(r.dispositivo.lei == "CF" for r in base.buscar("liberdade", lei="CF"))
assert base.leis_ausentes("art. 33 da Lei de Drogas") == ["L11343"]
assert base.buscar("art. 157 do CP")[0].motivo == "referência"

# ---------- API ----------
limite_fontes.limpar()
busca = cliente.get("/fontes/buscar", params={"q": "furto durante o repouso noturno"}).json()
assert busca["resultados"][0]["rotulo"] == "CP.art155" and busca["resultados"][0]["fonte"]["atualizado_ate"] == "2026-01"
assert "Lei de Drogas" in cliente.get("/fontes/buscar", params={"q": "art. 33 da Lei de Drogas"}).json()["avisos"][0]
leis = {lei["lei"]: lei for lei in cliente.get("/fontes/leis").json()}
assert leis["CC"]["fontes"][0]["desatualizada"] and not leis["CP"]["fontes"][0]["desatualizada"]
assert cliente.get("/fontes/dispositivo/CP.art155.§4.IV").json()["texto"] == "IV – mediante concurso de duas ou mais pessoas;"
assert cliente.get("/fontes/dispositivo/CP.art9999").status_code == 404
# entradas fora do formato: 422, sem chegar à busca
assert cliente.get("/fontes/buscar", params={"q": "x" * 301}).status_code == 422
assert cliente.get("/fontes/buscar", params={"q": "furto", "limite": 11}).status_code == 422
assert cliente.get("/fontes/buscar", params={"q": "furto", "lei": "cp; drop"}).status_code == 422
assert cliente.get("/fontes/dispositivo/CP.art155<script>").status_code == 422
# limite de requisições: 60 por minuto por visitante
limite_fontes.limpar()
codigos = [cliente.get("/fontes/buscar", params={"q": "furto"}).status_code for _ in range(61)]
assert codigos[:60] == [200] * 60 and codigos[60] == 429
limite_fontes.limpar()

# o cálculo traz o texto de cada dispositivo citado, conferido na base
limite_calculo.limpar()
calculo = cliente.post("/dosimetria/calcular", json=EXEMPLO_ENTRADA).json()
assert [(f["rotulo"], f["encontrado"]) for f in calculo["fontes_citadas"]] == [
    ("CP.art155", True), ("CP.art59", True), ("CP.art61.I", True), ("CP.art68", True), ("CP.art155.§1", True)
]
trafico = cliente.post("/dosimetria/calcular", json=cliente.get("/exemplos/construido-04").json()["entrada"]).json()
assert [f["rotulo"] for f in trafico["fontes_citadas"] if not f["encontrado"]][0].startswith("L11343")
limite_calculo.limpar()

print(f"Base de legislação: OK ({len(base.dispositivos)} artigos de {len(por_lei)} leis; busca: "
      + ", ".join(f"{k} {v:.0%}" for k, v in metricas.items() if k.startswith("acerto")) + f", MRR {metricas['mrr']:.2f})")

## Agente da aba Analisar caso (agente/)

A leitura das penas e das frações direto do texto da lei, a identificação do crime e as sugestões (casos de teste em `dados/agente/casos.json`), e as rotas `/agente/...`. Nada é gravado.

In [ ]:
from fractions import Fraction as F

from agente.analise import agente
from agente.avaliacao import avaliar as avaliar_agente
from agente.lei import ler_crime, ler_fracao, ler_pena
from api.agente import limite_agente

o_agente = agente()

# ---------- parser de penas: o texto da lei vira dias (1 ano = 365, 1 mês = 30) ----------
def faixa(rotulo):
    pena = o_agente.crimes[rotulo].pena
    return pena.especie, pena.minimo_dias, pena.maximo_dias, pena.multa

assert faixa("CP.art155") == ("reclusão", 365, 1460, True)   # critério de pronto da fase 1 do plano
assert faixa("CP.art157") == ("reclusão", 1460, 3650, True)
assert faixa("CP.art121") == ("reclusão", 2190, 7300, False)
assert faixa("CP.art129") == ("detenção", 90, 365, False)
assert faixa("CP.art213") == ("reclusão", 2190, 3650, False)
assert faixa("CP.art171") == ("reclusão", 365, 1825, True)
assert faixa("CP.art147") == ("detenção", 30, 180, True)
assert faixa("LCP.art21") == ("prisão simples", 15, 90, True)
assert ler_pena("Pena – reclusão, de 2 (dois) anos e 6 (seis) meses a 5 (cinco) anos").minimo_dias == 910  # 2 × 365 + 6 × 30
assert ler_pena("Pena – detenção, de quinze dias a seis meses, ou multa").maximo_dias == 180
assert ler_pena("Pena – multa.") is None
assert len(o_agente.crimes) >= 300, len(o_agente.crimes)

# ---------- parser de frações ----------
assert ler_fracao("A pena aumenta-se de 1/3 (um terço) até metade:") == ("aumento", F(1, 3), F(1, 2))
assert ler_fracao("o juiz pode ... diminuí-la de um a dois terços") == ("diminuicao", F(1, 3), F(2, 3))
assert ler_fracao("o juiz pode reduzir a pena de um sexto a um terço") == ("diminuicao", F(1, 6), F(1, 3))
assert ler_fracao("aplica-se em dobro a pena prevista no caput") == ("aumento", F(1), F(1))
assert ler_fracao("A pena aumenta-se de 2/3 (dois terços):") == ("aumento", F(2, 3), F(2, 3))
gerais = {c.codigo: f for c, f in o_agente._causas_gerais}
assert gerais["tentativa"] == ("diminuicao", F(1, 3), F(2, 3))
assert gerais["arrependimento_posterior"] == ("diminuicao", F(1, 3), F(2, 3))
assert gerais["participacao_de_menor_importancia"] == ("diminuicao", F(1, 6), F(1, 3))
assert gerais["concurso_formal"] == ("aumento", F(1, 6), F(1, 2))
assert gerais["crime_continuado"] == ("aumento", F(1, 6), F(2, 3))

# ---------- estrutura lida do artigo ----------
roubo = {o.rotulo: o for o in o_agente.crimes["CP.art157"].opcoes}
assert (roubo["CP.art157.§2.II"].fracao_min, roubo["CP.art157.§2.II"].fracao_max) == (F(1, 3), F(1, 2))
assert roubo["CP.art157.§2-B"].fracao_min == F(1)                       # arma de uso restrito: em dobro
assert roubo["CP.art157.§3.II"].pena.minimo_dias == 7300                # latrocínio: 20 a 30 anos
assert "CP.art157.§2.I" not in roubo, "inciso revogado não vira opção"
furto = {o.rotulo: o for o in o_agente.crimes["CP.art155"].opcoes}
assert furto["CP.art155.§4.IV"].tipo == "forma" and furto["CP.art155.§4.IV"].pena.minimo_dias == 730
assert furto["CP.art155.§1"].tipo == "aumento" and furto["CP.art155.§2"].tipo == "diminuicao"

# ---------- casos de teste do agente ----------
resultados_agente, metricas_agente = avaliar_agente(o_agente)
falhas = [(r.id, r.crimes, r.faltaram, r.indevidos) for r in resultados_agente if not r.passou]
assert not falhas, falhas

# o que a descrição não informa: "mata" já diz que o crime se consumou, "um homem com 23 anos" já dá a idade
feminicidio = "Por volta das 2 horas da manhã, um homem com 23 anos mata sua esposa com facadas, e não confessou."
assert o_agente.informacoes_ausentes(feminicidio) == ["antecedentes e reincidência do réu"]
# negação: "não confessou" não sugere a atenuante da confissão
atenuantes = {a["codigo"]: a for a in o_agente.estrutura("CP.art121-A", feminicidio)["atenuantes"]}
assert not atenuantes["confissao_espontanea"]["sugerido"]

# ---------- API do agente ----------
limite_agente.limpar()
caso = "Por volta das 2h da madrugada, o réu Rafael Souza, junto com um comparsa, arrombou a porta de uma loja e subtraiu dois celulares. Ele é reincidente. Confessou."
analise = cliente.post("/agente/analisar", json={"descricao": caso}).json()
assert analise["crimes"][0]["rotulo"] == "CP.art155"
assert "Rafael" not in json.dumps(analise, ensure_ascii=False), "a resposta só traz o texto anonimizado"
sugeridas = {i["rotulo"] for i in analise["estrutura"]["formas"] + analise["estrutura"]["causas"] if i["sugerido"]}
assert {"CP.art155.§4.I", "CP.art155.§4.IV"} <= sugeridas and "CP.art155.§1" not in sugeridas
assert "Tema 1.087" in next(i["observacao"] for i in analise["estrutura"]["causas"] if i["rotulo"] == "CP.art155.§1")
assert "tráfico" in " ".join(cliente.post("/agente/analisar", json={"descricao": "O réu vendia maconha e cocaína na porta da escola, em pequenas porções, para vários usuários."}).json()["avisos"])
assert cliente.post("/agente/analisar", json={"descricao": "curto"}).status_code == 422
assert cliente.post("/agente/analisar", json={"descricao": "x" * 20_001}).status_code == 422
assert cliente.post("/agente/estrutura", json={"crime": "CP.art9999"}).status_code == 422
assert cliente.post("/agente/estrutura", json={"crime": "<script>"}).status_code == 422
assert len(cliente.get("/agente/crimes").json()) >= 300

calculo = cliente.post("/agente/calcular", json={
    "crime": "CP.art155", "formas": ["CP.art155.§4.I", "CP.art155.§4.IV"], "causas": ["CP.art155.§1"],
    "agravantes": ["reincidencia"], "atenuantes": ["confissao_espontanea"],
}).json()
assert calculo["resultado"]["pena_definitiva"]["texto"] == "2 anos, 9 meses, 3 dias", calculo["resultado"]["pena_definitiva"]
assert calculo["entrada"]["faixa"]["origem"] == "CP.art155.§4.I" and calculo["entrada"]["circunstancias_desfavoraveis"] == ["circunstancias"]
assert calculo["entrada"]["causas"] == [], "repouso noturno não entra no furto qualificado (Tema 1.087)"
assert any("Tema 1.087" in a for a in calculo["alertas"]) and any("mais de uma qualificadora" in a for a in calculo["alertas"])
assert all(f["encontrado"] for f in calculo["resultado"]["fontes_citadas"])
# roubo com duas majorantes do § 2º: uma causa só, com o aviso da Súmula 443
roubo_calc = cliente.post("/agente/calcular", json={"crime": "CP.art157", "causas": ["CP.art157.§2.II", "CP.art157.§2.V", "CP.art157.§2-A.I"]}).json()
assert len(roubo_calc["entrada"]["causas"]) == 2 and any("Súmula 443" in a for a in roubo_calc["alertas"])
assert roubo_calc["resultado"]["alternativa_art68"] is not None
# bis in idem: motivo fútil qualificou o homicídio, não agrava de novo
homicidio = cliente.post("/agente/calcular", json={"crime": "CP.art121", "formas": ["CP.art121.§2.II"], "agravantes": ["motivo_futil_ou_torpe"]}).json()
assert homicidio["entrada"]["agravantes_atenuantes"] == [] and "bis in idem" in homicidio["alertas"][0]
# feminicídio: violência doméstica e contra cônjuge são elementares do tipo, não agravam de novo
femin = cliente.post("/agente/calcular", json={"crime": "CP.art121-A", "agravantes": ["relacoes_domesticas", "contra_familiar"]}).json()
assert femin["entrada"]["agravantes_atenuantes"] == [] and all("elementar" in a for a in femin["alertas"])
assert femin["resultado"]["pena_base"]["texto"] == "20 anos"
# itens de outro crime, códigos fora do formato e rótulos inventados: 422
assert cliente.post("/agente/calcular", json={"crime": "CP.art157", "formas": ["CP.art155.§4.I"]}).status_code == 422
assert cliente.post("/agente/calcular", json={"crime": "CP.art155", "agravantes": ["DROP TABLE"]}).status_code == 422
assert cliente.post("/agente/calcular", json={"crime": "CP.art155", "causas": ["CP.art155.§99"]}).status_code == 422
assert cliente.post("/agente/calcular", json={"crime": "CP.art155", "extra": 1}).status_code == 422
# limite de requisições: 30 por minuto por visitante
limite_agente.limpar()
codigos = [cliente.post("/agente/estrutura", json={"crime": "CP.art155"}).status_code for _ in range(31)]
assert codigos[:30] == [200] * 30 and codigos[30] == 429
limite_agente.limpar()

print(f"Agente: OK ({len(o_agente.crimes)} crimes lidos da lei; casos de teste: "
      + ", ".join(f"{k} {v:.0%}" for k, v in metricas_agente.items() if isinstance(v, float)) + ")")